# Merge all demographics with Philadelphia census tracts

In [153]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

## 2010 Tracts

### Demographic data preprocessing

In [154]:
# Load all demographic data
race_ethn_2010 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\race_ethn_2010.csv")
med_prop_val_2010 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\med_prop_val_2010.csv")
med_inc_2010 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\med_inc_2010.csv")
edu_2010 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\edu_2010.csv")
gent_YN_2010 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\gent_YN_2010.csv")

In [155]:
race_ethn_2010.head()

,Unnamed: 0,geography,geographic_area_name,tot_pop,TRACTCE,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only
0,0,1400000US42101000100,"Census Tract 1, Philadelphia County, Pennsylvania",3478,100.0,0.063830,0.125359,0.036228,0.774583
1,1,1400000US42101000200,"Census Tract 2, Philadelphia County, Pennsylvania",2937,200.0,0.102826,1.280899,0.026898,-0.410623
2,2,1400000US42101000300,"Census Tract 3, Philadelphia County, Pennsylvania",3169,300.0,0.107920,0.244241,0.042600,0.605238
3,3,1400000US42101000401,"Census Tract 4.01, Philadelphia County, Pennsy...",2125,401.0,0.186353,0.525176,0.050353,0.238118
4,4,1400000US42101000402,"Census Tract 4.02, Philadelphia County, Pennsy...",3142,402.0,0.056652,0.243794,0.034691,0.664863


In [156]:
# Fix GEO_ID
race_ethn_2010["GEO_ID"] = race_ethn_2010["geography"].str[9:]

In [157]:
med_prop_val_2010.head()

,Unnamed: 0,GEO_ID,med_prop_val,tract
0,1,1400000US42101000100,354800.0,1.00
1,2,1400000US42101000200,254700.0,2.00
2,3,1400000US42101000300,456000.0,3.00
3,4,1400000US42101000401,429600.0,4.01
4,5,1400000US42101000402,152800.0,4.02


In [158]:
# Fix GEO_ID
med_prop_val_2010["GEO_ID"] = med_prop_val_2010["GEO_ID"].str[9:]

In [159]:
med_inc_2010.head()

,Unnamed: 0,GEO_ID,med_inc,tract
0,1,1400000US42101000100,73041.0,1.00
1,2,1400000US42101000200,43218.0,2.00
2,3,1400000US42101000300,65577.0,3.00
3,4,1400000US42101000401,21832.0,4.01
4,5,1400000US42101000402,50020.0,4.02


In [160]:
# Fix GEO_ID
med_inc_2010["GEO_ID"] = med_inc_2010["GEO_ID"].str[9:]

In [161]:
edu_2010

,GEO_ID,geometry,total_25_plus,pct_bachelor_or_higher,pct_bachelor_lower
0,42101036301,"POLYGON ((2748366.299052715 290799.1266722697,...",2604,15.6,84.4
1,42101036400,POLYGON ((2743809.1538660973 294040.4015446496...,393,1.8,98.2
2,42101036600,POLYGON ((2699720.168483097 236062.07445901426...,1139,75.4,24.6
3,42101034803,POLYGON ((2735668.5653288523 276129.1361632435...,3305,19.9,80.1
4,42101034702,"POLYGON ((2730958.367196348 278534.1718410784,...",3300,28.3,71.7
...,...,...,...,...,...
379,42101037200,POLYGON ((2691718.702876651 223096.79226875928...,3085,20.5,79.5
380,42101038300,"POLYGON ((2706067.340606484 260203.5274151703,...",1657,1.4,98.6
381,42101039000,POLYGON ((2713663.2106345557 268770.6058941973...,4287,13.4,86.6
382,42101037800,"POLYGON ((2707568.841394347 242366.8674413042,...",1502,17.2,82.8


In [162]:
gent_YN_2010.head()

,OBJECTID,STATEFP10,COUNTYFP10,TRACTCE10,GEOID10,NAME10,NAMELSAD10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,LOGRECNO,SHAPE_AREA,SHAPE_LEN,GentYN
0,46,42,101,5000,42101005000,50.0,Census Tract 50,G5020,S,4504459.0,3340294.0,39.889877,-75.169141,10389,8.832943e+07,48033.335048,0
1,47,42,101,5400,42101005400,54.0,Census Tract 54,G5020,S,1846253.0,530013.0,39.890454,-75.251392,10390,2.575213e+07,24624.823144,0
2,48,42,101,5500,42101005500,55.0,Census Tract 55,G5020,S,1168442.0,12010.0,39.907419,-75.248917,10391,1.270364e+07,20389.059940,0
3,49,42,101,5600,42101005600,56.0,Census Tract 56,G5020,S,840411.0,0.0,39.898833,-75.244735,10392,9.047139e+06,11788.147571,0
4,50,42,101,6000,42101006000,60.0,Census Tract 60,G5020,S,1089657.0,0.0,39.911520,-75.238157,10393,1.175557e+07,14205.267812,0


In [163]:
gent_YN_2010["GEO_ID"] = gent_YN_2010["GEOID10"]

In [164]:
# Keep only desired columns
race_ethn_2010 = race_ethn_2010[["GEO_ID", "tot_pop", "prop_black", "prop_asian", "prop_hispanic_or_latino", "prop_white_only"]]
med_prop_val_2010 = med_prop_val_2010[["GEO_ID", "med_prop_val"]]
med_inc_2010 = med_inc_2010[["GEO_ID", "med_inc"]]
edu_2010 = edu_2010[["GEO_ID", "total_25_plus", "pct_bachelor_or_higher", "pct_bachelor_lower"]]
gent_YN_2010 = gent_YN_2010[["GEO_ID", "GentYN"]]

In [165]:
# Convert to GEO_IDs to numeric
race_ethn_2010["GEO_ID"] = pd.to_numeric(race_ethn_2010["GEO_ID"])
med_prop_val_2010["GEO_ID"] = pd.to_numeric(med_prop_val_2010["GEO_ID"])
med_inc_2010["GEO_ID"] = pd.to_numeric(med_inc_2010["GEO_ID"])
edu_2010["GEO_ID"] = pd.to_numeric(edu_2010["GEO_ID"])
gent_YN_2010["GEO_ID"] = pd.to_numeric(gent_YN_2010["GEO_ID"])

In [166]:
# Merge demographic data into one dataframe
demog_2010 = race_ethn_2010.merge(med_prop_val_2010, on="GEO_ID") \
              .merge(med_inc_2010, on="GEO_ID") \
              .merge(edu_2010, on="GEO_ID") \
              .merge(gent_YN_2010, on="GEO_ID", how="outer")

In [167]:
demog_2010.head()

,GEO_ID,tot_pop,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only,med_prop_val,med_inc,total_25_plus,pct_bachelor_or_higher,pct_bachelor_lower,GentYN
0,42101000100,3478.0,0.063830,0.125359,0.036228,0.774583,354800.0,73041.0,2752.0,72.3,27.7,0
1,42101000200,2937.0,0.102826,1.280899,0.026898,-0.410623,254700.0,43218.0,1602.0,33.3,66.7,0
2,42101000300,3169.0,0.107920,0.244241,0.042600,0.605238,456000.0,65577.0,2551.0,68.8,31.2,0
3,42101000401,2125.0,0.186353,0.525176,0.050353,0.238118,429600.0,21832.0,1247.0,71.5,28.5,0
4,42101000402,3142.0,0.056652,0.243794,0.034691,0.664863,152800.0,50020.0,2942.0,58.9,41.1,0


In [168]:
demog_2010.info()

<class 'pandas.DataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   GEO_ID                   384 non-null    int64  
 1   tot_pop                  381 non-null    float64
 2   prop_black               381 non-null    float64
 3   prop_asian               381 non-null    float64
 4   prop_hispanic_or_latino  381 non-null    float64
 5   prop_white_only          381 non-null    float64
 6   med_prop_val             367 non-null    float64
 7   med_inc                  374 non-null    float64
 8   total_25_plus            381 non-null    float64
 9   pct_bachelor_or_higher   376 non-null    float64
 10  pct_bachelor_lower       376 non-null    float64
 11  GentYN                   384 non-null    int64  
dtypes: float64(10), int64(2)
memory usage: 36.1 KB


### Merging demographic data with tracts geodataframe

In [169]:
# load 2010 tracts
tracts_2010 = gpd.read_file("C:\\UC San Diego\\GPEC447\\FinalProject\\tl_2010_42101_tract10\\tl_2010_42101_tract10.shp")
tracts_2010.to_crs(2272, inplace=True)
tracts_2010["GEOID10"] = pd.to_numeric(tracts_2010["GEOID10"])

In [170]:
tracts_2010.head()

,STATEFP10,COUNTYFP10,TRACTCE10,GEOID10,NAME10,NAMELSAD10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,geometry
0,42,101,036301,42101036301,363.01,Census Tract 363.01,G5020,S,2322732,66075,+40.0895349,-074.9667387,"POLYGON ((2748366.299 290799.127, 2748435.97 2..."
1,42,101,036400,42101036400,364,Census Tract 364,G5020,S,4501110,8014,+40.1127747,-074.9789137,"POLYGON ((2743809.154 294040.402, 2743494.584 ..."
2,42,101,036600,42101036600,366,Census Tract 366,G5020,S,1004313,1426278,+39.9470272,-075.1404472,"POLYGON ((2699720.168 236062.074, 2699719.035 ..."
3,42,101,034803,42101034803,348.03,Census Tract 348.03,G5020,S,1271533,8021,+40.0619427,-075.0023705,"POLYGON ((2735668.565 276129.136, 2735577.548 ..."
4,42,101,034702,42101034702,347.02,Census Tract 347.02,G5020,S,1016206,0,+40.0570427,-075.0283288,"POLYGON ((2730958.367 278534.172, 2730985.876 ..."


In [171]:
tracts_2010.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   STATEFP10   384 non-null    str     
 1   COUNTYFP10  384 non-null    str     
 2   TRACTCE10   384 non-null    str     
 3   GEOID10     384 non-null    int64   
 4   NAME10      384 non-null    str     
 5   NAMELSAD10  384 non-null    str     
 6   MTFCC10     384 non-null    str     
 7   FUNCSTAT10  384 non-null    str     
 8   ALAND10     384 non-null    int64   
 9   AWATER10    384 non-null    int64   
 10  INTPTLAT10  384 non-null    str     
 11  INTPTLON10  384 non-null    str     
 12  geometry    384 non-null    geometry
dtypes: geometry(1), int64(3), str(9)
memory usage: 61.7 KB


In [172]:
# Merge demographic data with tract shapefile
tracts_demog_2010 = tracts_2010.merge(demog_2010, left_on='GEOID10', right_on='GEO_ID')

In [173]:
tracts_demog_2010.head()

,STATEFP10,COUNTYFP10,TRACTCE10,GEOID10,NAME10,NAMELSAD10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,...,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only,med_prop_val,med_inc,total_25_plus,pct_bachelor_or_higher,pct_bachelor_lower,GentYN
0,42,101,036301,42101036301,363.01,Census Tract 363.01,G5020,S,2322732,66075,...,0.043572,0.191610,0.066576,0.698241,201900.0,54569.0,2604.0,15.6,84.4,0
1,42,101,036400,42101036400,364,Census Tract 364,G5020,S,4501110,8014,...,0.216216,0.028450,0.024182,0.731152,NaN,NaN,393.0,1.8,98.2,0
2,42,101,036600,42101036600,366,Census Tract 366,G5020,S,1004313,1426278,...,0.065733,0.227632,0.038953,0.667681,436500.0,130139.0,1139.0,75.4,24.6,0
3,42,101,034803,42101034803,348.03,Census Tract 348.03,G5020,S,1271533,8021,...,0.068109,0.057859,0.052392,0.821640,208000.0,56667.0,3305.0,19.9,80.1,0
4,42,101,034702,42101034702,347.02,Census Tract 347.02,G5020,S,1016206,0,...,0.073286,0.097189,0.064618,0.764907,219800.0,69981.0,3300.0,28.3,71.7,0


In [174]:
tracts_demog_2010.columns

Index(['STATEFP10', 'COUNTYFP10', 'TRACTCE10', 'GEOID10', 'NAME10',
       'NAMELSAD10', 'MTFCC10', 'FUNCSTAT10', 'ALAND10', 'AWATER10',
       'INTPTLAT10', 'INTPTLON10', 'geometry', 'GEO_ID', 'tot_pop',
       'prop_black', 'prop_asian', 'prop_hispanic_or_latino',
       'prop_white_only', 'med_prop_val', 'med_inc', 'total_25_plus',
       'pct_bachelor_or_higher', 'pct_bachelor_lower', 'GentYN'],
      dtype='str')

In [175]:
# Keep only relevant columns
tracts_demog_2010 = tracts_demog_2010[['geometry', 'GEO_ID', 'tot_pop',
       'prop_black', 'prop_asian', 'prop_hispanic_or_latino',
       'prop_white_only', 'med_prop_val', 'med_inc', 'total_25_plus',
       'pct_bachelor_or_higher', 'pct_bachelor_lower', 'GentYN']]

In [176]:
tracts_demog_2010.head()

,geometry,GEO_ID,tot_pop,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only,med_prop_val,med_inc,total_25_plus,pct_bachelor_or_higher,pct_bachelor_lower,GentYN
0,"POLYGON ((2748366.299 290799.127, 2748435.97 2...",42101036301,3695.0,0.043572,0.191610,0.066576,0.698241,201900.0,54569.0,2604.0,15.6,84.4,0
1,"POLYGON ((2743809.154 294040.402, 2743494.584 ...",42101036400,703.0,0.216216,0.028450,0.024182,0.731152,NaN,NaN,393.0,1.8,98.2,0
2,"POLYGON ((2699720.168 236062.074, 2699719.035 ...",42101036600,1643.0,0.065733,0.227632,0.038953,0.667681,436500.0,130139.0,1139.0,75.4,24.6,0
3,"POLYGON ((2735668.565 276129.136, 2735577.548 ...",42101034803,4390.0,0.068109,0.057859,0.052392,0.821640,208000.0,56667.0,3305.0,19.9,80.1,0
4,"POLYGON ((2730958.367 278534.172, 2730985.876 ...",42101034702,3807.0,0.073286,0.097189,0.064618,0.764907,219800.0,69981.0,3300.0,28.3,71.7,0


In [177]:
tracts_demog_2010.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   geometry                 384 non-null    geometry
 1   GEO_ID                   384 non-null    int64   
 2   tot_pop                  381 non-null    float64 
 3   prop_black               381 non-null    float64 
 4   prop_asian               381 non-null    float64 
 5   prop_hispanic_or_latino  381 non-null    float64 
 6   prop_white_only          381 non-null    float64 
 7   med_prop_val             367 non-null    float64 
 8   med_inc                  374 non-null    float64 
 9   total_25_plus            381 non-null    float64 
 10  pct_bachelor_or_higher   376 non-null    float64 
 11  pct_bachelor_lower       376 non-null    float64 
 12  GentYN                   384 non-null    int64   
dtypes: float64(10), geometry(1), int64(2)
memory usage: 39.1 

## 2020 Tracts

### Demographic data preprocessing

In [178]:
# Load all demographic data
race_ethn_2020 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\race_ethn_2020.csv")
med_prop_val_2020 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\med_prop_val_2020.csv")
med_inc_2020 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\med_inc_2020.csv")
edu_2020 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\edu_2020.csv")
gent_YN_2020 = pd.read_csv("C:\\UC San Diego\\GPEC447\\FinalProject\\gent_YN_2020.csv")

In [179]:
# Fix GEO_IDs
race_ethn_2020["GEO_ID"] = race_ethn_2020["geography"].str[9:]
med_prop_val_2020["GEO_ID"] = med_prop_val_2020["GEO_ID"].str[9:]
med_inc_2020["GEO_ID"] = med_inc_2020["GEO_ID"].str[9:]
gent_YN_2020["GEO_ID"] = gent_YN_2020["GEOID"]

In [180]:
# Keep only desired columns
race_ethn_2020 = race_ethn_2020[["GEO_ID", "tot_pop", "prop_black", "prop_asian", "prop_hispanic_or_latino", "prop_white_only"]]
med_prop_val_2020 = med_prop_val_2020[["GEO_ID", "med_prop_val"]]
med_inc_2020 = med_inc_2020[["GEO_ID", "med_inc"]]
edu_2020 = edu_2020[["GEO_ID", "total_25_plus", "pct_bachelor_or_higher", "pct_bachelor_lower"]]
gent_YN_2020 = gent_YN_2020[["GEO_ID", "GentYN"]]

In [181]:
# Convert to GEO_IDs to numeric
race_ethn_2020["GEO_ID"] = pd.to_numeric(race_ethn_2020["GEO_ID"])
med_prop_val_2020["GEO_ID"] = pd.to_numeric(med_prop_val_2020["GEO_ID"])
med_inc_2020["GEO_ID"] = pd.to_numeric(med_inc_2020["GEO_ID"])
edu_2020["GEO_ID"] = pd.to_numeric(edu_2020["GEO_ID"])
gent_YN_2020["GEO_ID"] = pd.to_numeric(gent_YN_2020["GEO_ID"])

In [182]:
# Merge demographic data into one dataframe
demog_2020 = race_ethn_2020.merge(med_prop_val_2020, on="GEO_ID") \
              .merge(med_inc_2020, on="GEO_ID") \
              .merge(edu_2020, on="GEO_ID") \
              .merge(gent_YN_2020, on="GEO_ID")

In [183]:
demog_2020.head()

,GEO_ID,tot_pop,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only,med_prop_val,med_inc,total_25_plus,pct_bachelor_or_higher,pct_bachelor_lower,GentYN
0,42101000101,2329,0.054959,0.101331,0.055818,0.787892,93462.0,104458.0,1708,91.627635,8.372365,0
1,42101000102,3511,0.050698,0.100826,0.059242,0.789234,98400.0,104236.0,2549,81.796783,18.203217,0
2,42101000200,3367,0.084942,0.634096,0.037125,0.243837,190300.0,83854.0,2526,50.237530,49.762470,0
3,42101000300,4501,0.080427,0.158631,0.061986,0.698956,75611.0,84843.0,2924,82.147743,17.852257,0
4,42101000401,3123,0.089657,0.276977,0.053474,0.579891,145592.0,73438.0,2297,81.802351,18.197649,0


In [184]:
demog_2020.info()

<class 'pandas.DataFrame'>
RangeIndex: 408 entries, 0 to 407
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   GEO_ID                   408 non-null    int64  
 1   tot_pop                  408 non-null    int64  
 2   prop_black               403 non-null    float64
 3   prop_asian               403 non-null    float64
 4   prop_hispanic_or_latino  403 non-null    float64
 5   prop_white_only          403 non-null    float64
 6   med_prop_val             379 non-null    float64
 7   med_inc                  383 non-null    float64
 8   total_25_plus            408 non-null    int64  
 9   pct_bachelor_or_higher   390 non-null    float64
 10  pct_bachelor_lower       390 non-null    float64
 11  GentYN                   408 non-null    int64  
dtypes: float64(8), int64(4)
memory usage: 38.4 KB


### Merging demographic data with tracts geodataframe

In [185]:
# load 2020 tracts
tracts_2020 = gpd.read_file("C:\\UC San Diego\\GPEC447\\FinalProject\\tl_2020_42_tract\\tl_2020_42_tract.shp")
tracts_2020.to_crs(2272, inplace=True)
tracts_2020["GEOID"] = pd.to_numeric(tracts_2020["GEOID"])

In [186]:
# Merge demographic data with tract shapefile
tracts_demog_2020 = tracts_2020.merge(demog_2020, left_on='GEOID', right_on='GEO_ID')

In [187]:
tracts_demog_2020.columns

Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'NAME', 'NAMELSAD', 'MTFCC',
       'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry',
       'GEO_ID', 'tot_pop', 'prop_black', 'prop_asian',
       'prop_hispanic_or_latino', 'prop_white_only', 'med_prop_val', 'med_inc',
       'total_25_plus', 'pct_bachelor_or_higher', 'pct_bachelor_lower',
       'GentYN'],
      dtype='str')

In [188]:
# Keep only relevant columns
tracts_demog_2020 = tracts_demog_2020[['geometry', 'GEO_ID', 'tot_pop',
       'prop_black', 'prop_asian', 'prop_hispanic_or_latino',
       'prop_white_only', 'med_prop_val', 'med_inc', 'total_25_plus',
       'pct_bachelor_or_higher', 'pct_bachelor_lower', 'GentYN']]

In [189]:
tracts_demog_2020.head()

,geometry,GEO_ID,tot_pop,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only,med_prop_val,med_inc,total_25_plus,pct_bachelor_or_higher,pct_bachelor_lower,GentYN
0,"POLYGON ((2676981.178 254440.769, 2677375.454 ...",42101012204,3694,0.728479,0.050893,0.041960,0.178668,27245.0,56495.0,3219,44.920783,55.079217,0
1,"POLYGON ((2678861.975 255056.314, 2678866.419 ...",42101012203,658,0.255319,0.256839,0.048632,0.439210,NaN,52209.0,626,65.495208,34.504792,0
2,"POLYGON ((2686906.79 243645.714, 2686997.702 2...",42101013602,3776,0.071769,0.062500,0.049523,0.816208,27378.0,92212.0,3269,83.328235,16.671765,0
3,"POLYGON ((2724650.476 284639.902, 2724859.044 ...",42101034502,5928,0.207659,0.191464,0.068657,0.532220,33666.0,47714.0,3940,41.573604,58.426396,0
4,"POLYGON ((2694530.584 234145.037, 2694537.691 ...",42101000902,2842,0.065799,0.245602,0.034131,0.654469,260311.0,61312.0,2251,69.258108,30.741892,0


In [190]:
tracts_demog_2020.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 408 entries, 0 to 407
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   geometry                 408 non-null    geometry
 1   GEO_ID                   408 non-null    int64   
 2   tot_pop                  408 non-null    int64   
 3   prop_black               403 non-null    float64 
 4   prop_asian               403 non-null    float64 
 5   prop_hispanic_or_latino  403 non-null    float64 
 6   prop_white_only          403 non-null    float64 
 7   med_prop_val             379 non-null    float64 
 8   med_inc                  383 non-null    float64 
 9   total_25_plus            408 non-null    int64   
 10  pct_bachelor_or_higher   390 non-null    float64 
 11  pct_bachelor_lower       390 non-null    float64 
 12  GentYN                   408 non-null    int64   
dtypes: float64(8), geometry(1), int64(4)
memory usage: 41.6 K

## Export

In [191]:
tracts_demog_2010.to_file("C:\\UC San Diego\\GPEC447\\FinalProject\\tracts_demog_2010.geojson", driver="GeoJSON")
tracts_demog_2020.to_file("C:\\UC San Diego\\GPEC447\\FinalProject\\tracts_demog_2020.geojson", driver="GeoJSON")